# 시험 8 — LoRA 켬/끔 비교: 같은 입력·같은 설정에서 나카무라 이펙트 LoRA가 실제로 화풍을 바꾸는지 (Kaggle·Colab T4, 학습 없음, 약 15–25분)
GPT 판정(docs/GPT-RESEARCH-MOTION-2026-10-26.md): 시험 4 결과만으로 LoRA가 화풍을 입혔는지 단정할 수 없다 → 같은 입력에서 LoRA 끔/켬, 강도 설정을 나눠 본다.
- 입력: 각진 폭발 블로킹(`fx_blocking.py --angular`)의 불만 검은 바탕에 둔 그림, 장 0·3·6·9·12 다섯 장.
- 설정 두 가지 × LoRA 끔/켬: **A** 강도 1.0 + 윤곽 0.7, **C** 강도 0.85 + 윤곽 0.6. 글·시드(1234) 동일, LoRA는 합치지 않고 켜고 끄기만.
- 결과: `motion8_sheet.jpg`(줄 = 입력 / A 끔 / A 켬 / C 끔 / C 켬, 칸 = 장) + 장별 그림.
- 설정(Kaggle): **GPU T4 x2**, **Internet On** → Save & Run All. Colab: T4 GPU → 모두 실행.
- 끝나면 `motion8_result.zip`을 Drive 링크로 알려 준다.


In [ ]:
# 0) 어디서 돌리나: Kaggle이면 /kaggle/working, Colab이면 /content/out 에 결과를 쓴다
import os, glob
COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
OUT = "/content/out" if COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
os.makedirs(OUT, exist_ok=True); print("결과 폴더", OUT)
# 1) 패키지와 자료: 기본 모델(HF), LoRA 3차 3바퀴·블로킹 두 가지(Drive)
# 버전 고정: Colab·Kaggle 기본 파이썬 3.13 + 새 transformers에서 diffusers 0.31이 깨졌다(FLAX_WEIGHTS_NAME, 2026-10-26). 서버 py3.13에서 아래 조합 import 확인
!pip install -q "diffusers==0.35.1" "transformers==4.56.2" "peft==0.17.1" "accelerate>=1.0" opencv-python-headless
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/lora.safetensors "https://drive.usercontent.google.com/download?id=1ht_UqE8JkJTUX-eO5lNIqvhyFRcprUWJ&export=download&confirm=t"
os.system('curl -sSL -o /tmp/blk7.zip "https://drive.usercontent.google.com/download?id=1ZkTCeGVrhy1Alg898iwKDPjObEA5VHRI&export=download&confirm=t"')
os.system("mkdir -p /tmp/blk7 && cd /tmp/blk7 && unzip -q -o /tmp/blk7.zip")
print(len(glob.glob("/tmp/blk7/ang/*.png")), "장")
!ls -la /tmp/ilxl.safetensors /tmp/lora.safetensors


In [ ]:
# 2) 파이프라인: SDXL ControlNet 그림→그림 + LoRA 합치기 (윤곽 조절 xinsir/controlnet-canny-sdxl-1.0)
import torch
from diffusers import StableDiffusionXLControlNetImg2ImgPipeline, ControlNetModel, EulerAncestralDiscreteScheduler
from PIL import Image
cn = ControlNetModel.from_pretrained("xinsir/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_single_file("/tmp/ilxl.safetensors", controlnet=cn, torch_dtype=torch.float16)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("/tmp/lora.safetensors", adapter_name="fx")
# 합치지 않는다: 같은 파이프라인에서 enable_lora/disable_lora로 켜고 끈다
pipe.to("cuda")
pipe.vae.to(torch.float32)
print(torch.cuda.memory_allocated() / 1e9, "GB")


In [ ]:
# 3) LoRA 끔/켬 × 설정 두 가지, 장 다섯 개
import numpy as np, json, cv2
P = "nkmr_fx, vivid colors, anime screencap, explosion, fire, embers, white core orange rim, layered cel shading, no outline, sharp tips, large effect, no humans, black background, simple background"
N = "lowres, worst quality, text, watermark, city, building, scenery, sky, ground, horizon"
STEPS, CFG, SEED = 28, 6.0, 1234
KEYS = [0, 3, 6, 9, 12]
def fxmask(b):
    h = cv2.cvtColor(b, cv2.COLOR_RGB2HSV).astype(int)
    H, S, V = h[..., 0], h[..., 1], h[..., 2]
    return ((((H < 35) | (H > 165)) & (S > 70) & (V > 70)) | ((V > 200) & (S < 90))).astype(np.uint8)
def edges(a):
    e = cv2.Canny(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), 30, 90)
    return Image.fromarray(np.stack([e] * 3, -1))
def run(a, s, cs):
    g = torch.Generator("cuda").manual_seed(SEED)
    with torch.inference_mode():
        return pipe(prompt=P, negative_prompt=N, image=Image.fromarray(a), control_image=edges(a), strength=s, controlnet_conditioning_scale=cs,
                    num_inference_steps=STEPS, guidance_scale=CFG, generator=g, width=1344, height=768).images[0]
fs = sorted(glob.glob("/tmp/blk7/ang/*.png"))
ins = []
for k in KEYS:
    b = np.asarray(Image.open(fs[k]).convert("RGB"))
    ins.append((b * fxmask(b)[..., None]).astype(np.uint8))
rows = {"input": [Image.fromarray(a) for a in ins]}
for tag, s, cs in (("A", 1.0, 0.7), ("C", 0.85, 0.6)):
    for lo in ("off", "on"):
        pipe.enable_lora() if lo == "on" else pipe.disable_lora()
        rows[f"{tag}_lora_{lo}"] = [run(a, s, cs) for a in ins]
        print(tag, lo, "끝")
for k, ims in rows.items():
    d = f"{OUT}/m8/{k}"; os.makedirs(d, exist_ok=True)
    for kk, im in zip(KEYS, ims):
        im.save(f"{d}/f{kk:03d}.png")


In [ ]:
# 4) 비교 시트: 줄 = 입력 / A 끔 / A 켬 / C 끔 / C 켬
def lab(im, t):
    a = cv2.resize(cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2BGR), (448, 256)); cv2.putText(a, t, (6, 22), 0, 0.6, (255, 255, 255), 2); return a
sheet = np.vstack([np.hstack([lab(im, f"{k} f{kk}") for kk, im in zip(KEYS, ims)]) for k, ims in rows.items()])
cv2.imwrite(f"{OUT}/motion8_sheet.jpg", sheet, [cv2.IMWRITE_JPEG_QUALITY, 90])
!cd {OUT} && zip -q -r motion8_result.zip m8 motion8_sheet.jpg && ls -la {OUT}


In [ ]:
# 5) Colab이면 결과 zip 내려받기(Kaggle은 Output 탭에서 받음)
if COLAB and OUT.startswith("/content"):
    from google.colab import files
    files.download(f"{OUT}/motion8_result.zip")
